# Taller Sesión 2: Fine-tuning de Qwen2.5-4B con LoRA/Unsloth

Este notebook te guiará paso a paso en el fine-tuning de un LLM usando técnicas eficientes:
- **Modelo**: Qwen2.5-4B-Instruct (4 mil millones de parámetros)
- **Técnica**: LoRA (Low-Rank Adaptation)
- **Acelerador**: Unsloth para training 2x más rápido
- **Caso de uso**: Chatbot especializado en atención al cliente

**Requisitos**:
- GPU con 12-16GB VRAM (RTX 3060, 4060 Ti, o superior)
- Python 3.10+
- CUDA 11.8 o superior

## 1. Instalación de Dependencias

Instalamos Unsloth, Transformers y las bibliotecas necesarias.

In [ ]:
# Instalar Unsloth y dependencias
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Verificar instalación y GPU
import torch
import unsloth

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Unsloth version: {unsloth.__version__}")

## 2. Carga del Modelo Base con Unsloth

Cargamos Qwen2.5-4B-Instruct en formato 4-bit (QLoRA) para reducir uso de VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuración del modelo
max_seq_length = 2048  # Longitud máxima de secuencia
dtype = None  # Auto-detección (Float16 para T4, V100, Bfloat16 para Ampere+)
load_in_4bit = True  # Usar cuantización 4-bit para ahorrar VRAM

# Cargar modelo y tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",  # Modelo pre-optimizado por Unsloth
    # Si quieres el 4B original: "Qwen/Qwen2.5-4B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Modelo cargado exitosamente!")
print(f"Parámetros del modelo: {model.num_parameters() / 1e9:.2f}B")

## 3. Configuración de LoRA

Añadimos adaptadores LoRA al modelo. Solo estos adaptadores serán entrenados.

In [ ]:
# Configurar LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Rank de LoRA (4, 8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],  # Capas a adaptar
    lora_alpha = 16,  # Factor de escalado (típicamente igual a r)
    lora_dropout = 0,  # Dropout (0 es óptimo para Unsloth)
    bias = "none",  # No entrenar bias
    use_gradient_checkpointing = "unsloth",  # Optimización de memoria
    random_state = 3407,  # Seed para reproducibilidad
    use_rslora = False,  # Rank-Stabilized LoRA
    loftq_config = None,  # LoftQ inicialización
)

# Ver parámetros entrenables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"Parámetros totales: {all_params:,}")

## 4. Preparación del Dataset

Creamos un dataset de ejemplo para atención al cliente. En producción, usarías tus propios datos.

In [ ]:
# Formato de prompts para Qwen2.5
alpaca_prompt = """A continuación hay una instrucción que describe una tarea, junto con una entrada que proporciona más contexto. Escribe una respuesta que complete apropiadamente la solicitud.

### Instrucción:
{}

### Entrada:
{}

### Respuesta:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Token de fin de secuencia

def formatting_prompts_func(examples):
    """Formatea ejemplos en el formato Alpaca."""
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Dataset de ejemplo: Atención al cliente de telecomunicaciones
dataset_examples = [
    {
        "instruction": "Responde a la consulta del cliente de manera profesional y útil.",
        "input": "Mi internet está muy lento desde ayer, ¿qué puedo hacer?",
        "output": "Entiendo tu frustración con la velocidad del internet. Vamos a resolverlo paso a paso:\n1. Reinicia tu router desconectándolo 30 segundos\n2. Verifica que no haya muchos dispositivos conectados\n3. Prueba la velocidad en speedtest.net\nSi el problema persiste, puedo enviarte un técnico sin costo. ¿Cuál es tu número de cuenta?"
    },
    {
        "instruction": "Responde a la consulta del cliente de manera profesional y útil.",
        "input": "¿Cómo puedo cambiar mi plan a uno con más datos?",
        "output": "Con gusto te ayudo a mejorar tu plan. Tenemos las siguientes opciones:\n- Plan Plus: 50GB + llamadas ilimitadas - $299/mes\n- Plan Premium: 100GB + llamadas + Netflix - $399/mes\n- Plan Ultra: Datos ilimitados + streaming - $499/mes\nEl cambio es inmediato y pagas el proporcional este mes. ¿Te interesa alguno?"
    },
    {
        "instruction": "Responde a la consulta del cliente de manera profesional y útil.",
        "input": "No puedo hacer llamadas, sale error de red",
        "output": "Lamento el inconveniente. Vamos a solucionarlo:\n1. Activa y desactiva el modo avión\n2. Verifica que tengas señal (barras en pantalla)\n3. Reinicia tu teléfono\n4. Revisa que tu saldo esté activo\nSi nada funciona, puede ser una falla de red en tu zona. Déjame verificar tu ubicación y te confirmo en 5 minutos."
    },
    {
        "instruction": "Responde a la consulta del cliente de manera profesional y útil.",
        "input": "Quiero cancelar mi servicio",
        "output": "Lamento que desees cancelar. Antes de proceder, ¿puedo saber si hay algo que no esté funcionando bien? Quizás podamos resolverlo o ajustar tu plan para que se adapte mejor a tus necesidades. Si aún deseas cancelar, necesitaré:\n- Número de cuenta\n- Identificación oficial\n- Confirmación escrita\n¿Podemos revisar primero las opciones?"
    },
    {
        "instruction": "Responde a la consulta del cliente de manera profesional y útil.",
        "input": "Mi factura vino más alta este mes",
        "output": "Revisemos tu factura juntos. Los cargos adicionales pueden ser por:\n- Exceso de datos: $5 por GB extra\n- Llamadas internacionales\n- Servicios de valor agregado\n- Ajuste proporcional de plan\nDéjame ver tu cuenta... [simula espera] Veo que este mes consumiste 15GB extras. ¿Quieres que te cambie a un plan con más datos para evitar estos cargos?"
    },
]

# IMPORTANTE: En producción, carga tu dataset real
# from datasets import load_dataset
# dataset = load_dataset("json", data_files="tu_dataset.jsonl", split="train")

# Para este ejemplo, creamos un dataset pequeño
from datasets import Dataset
dataset = Dataset.from_list(dataset_examples * 20)  # Replicamos para tener más ejemplos
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset preparado: {len(dataset)} ejemplos")
print("\nEjemplo de formato:")
print(dataset[0]["text"][:500] + "...")

## 5. Configuración del Entrenamiento

Definimos los hiperparámetros y configuramos el Trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Puede hacer training 5x más rápido para secuencias cortas
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,  # Número de épocas
        max_steps = -1,  # -1 = entrenar todas las épocas
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",  # Cambiar a "wandb" si usas Weights & Biases
    ),
)

print("Trainer configurado. Iniciando entrenamiento...")
print(f"Batch size efectivo: {2 * 4} (per_device * gradient_accumulation)")
print(f"Total steps: ~{len(dataset) * 3 // (2 * 4)} steps")

## 6. Entrenamiento (Fine-tuning)

¡Momento de entrenar! Este proceso tomará entre 10-30 minutos dependiendo de tu GPU.

In [ ]:
# Mostrar estadísticas de GPU antes del entrenamiento
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"VRAM en uso antes del training: {start_gpu_memory} GB.")

# Entrenar
trainer_stats = trainer.train()

# Estadísticas finales
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\n{'='*50}")
print(f"Entrenamiento completado!")
print(f"{'='*50}")
print(f"Tiempo total: {trainer_stats.metrics['train_runtime']:.2f} segundos")
print(f"VRAM peak usada: {used_memory} GB")
print(f"VRAM para LoRA: {used_memory_for_lora} GB")
print(f"Porcentaje usado: {used_percentage}%")
print(f"Loss final: {trainer_stats.metrics['train_loss']:.4f}")

## 7. Inferencia con el Modelo Fine-tuned

Probemos el modelo con nuevas consultas.

In [ ]:
# Habilitar modo de inferencia rápida
FastLanguageModel.for_inference(model)

def generar_respuesta(instruccion, input_texto):
    """Genera una respuesta usando el modelo fine-tuned."""
    prompt = alpaca_prompt.format(
        instruccion,
        input_texto,
        ""  # Respuesta vacía, el modelo la completará
    )
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        temperature = 0.7,
        top_p = 0.9,
        top_k = 50,
        use_cache = True
    )
    
    # Decodificar solo la respuesta generada (sin el prompt)
    resultado_completo = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraer solo la respuesta
    if "### Respuesta:" in resultado_completo:
        respuesta = resultado_completo.split("### Respuesta:")[1].strip()
    else:
        respuesta = resultado_completo
    
    return respuesta

print("Modelo listo para inferencia!")

In [ ]:
# Prueba 1: Nueva consulta
print("="*60)
print("PRUEBA 1: Internet lento")
print("="*60)

consulta1 = "¿Por qué mi WiFi no funciona bien en el segundo piso?"
respuesta1 = generar_respuesta(
    "Responde a la consulta del cliente de manera profesional y útil.",
    consulta1
)

print(f"Cliente: {consulta1}")
print(f"\nAsistente: {respuesta1}")

In [ ]:
# Prueba 2: Consulta de facturación
print("\n" + "="*60)
print("PRUEBA 2: Facturación")
print("="*60)

consulta2 = "Me cobraron de más este mes, quiero un reembolso"
respuesta2 = generar_respuesta(
    "Responde a la consulta del cliente de manera profesional y útil.",
    consulta2
)

print(f"Cliente: {consulta2}")
print(f"\nAsistente: {respuesta2}")

In [ ]:
# Prueba 3: Consulta técnica
print("\n" + "="*60)
print("PRUEBA 3: Soporte técnico")
print("="*60)

consulta3 = "Mi teléfono no se conecta a los datos móviles"
respuesta3 = generar_respuesta(
    "Responde a la consulta del cliente de manera profesional y útil.",
    consulta3
)

print(f"Cliente: {consulta3}")
print(f"\nAsistente: {respuesta3}")

## 8. Comparación: Modelo Base vs Fine-tuned

Comparemos cómo responde el modelo base vs nuestro modelo fine-tuned.

In [ ]:
# Cargar modelo base para comparación
print("Cargando modelo base para comparación...")

model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model_base)

def comparar_modelos(consulta):
    """Compara respuestas del modelo base vs fine-tuned."""
    
    instruccion = "Responde a la consulta del cliente de manera profesional y útil."
    prompt = alpaca_prompt.format(instruccion, consulta, "")
    
    # Respuesta del modelo BASE
    inputs_base = tokenizer_base([prompt], return_tensors="pt").to("cuda")
    outputs_base = model_base.generate(**inputs_base, max_new_tokens=200, temperature=0.7)
    respuesta_base = tokenizer_base.decode(outputs_base[0], skip_special_tokens=True)
    if "### Respuesta:" in respuesta_base:
        respuesta_base = respuesta_base.split("### Respuesta:")[1].strip()
    
    # Respuesta del modelo FINE-TUNED
    respuesta_finetuned = generar_respuesta(instruccion, consulta)
    
    print(f"{'='*70}")
    print(f"CONSULTA: {consulta}")
    print(f"{'='*70}")
    print(f"\n🔹 MODELO BASE:")
    print(respuesta_base)
    print(f"\n✨ MODELO FINE-TUNED:")
    print(respuesta_finetuned)
    print(f"\n{'='*70}\n")

# Comparar en una consulta de ejemplo
comparar_modelos("Mi internet está muy lento, ¿qué hago?")

## 9. Guardar el Modelo

Guardamos el modelo fine-tuned para uso futuro.

In [ ]:
# Opción 1: Guardar solo los adaptadores LoRA (RECOMENDADO - muy ligero)
model.save_pretrained("qwen_customer_support_lora")  # Solo ~20-50MB
tokenizer.save_pretrained("qwen_customer_support_lora")

print("✅ Adaptadores LoRA guardados en: qwen_customer_support_lora/")
print("Para cargar: model = FastLanguageModel.from_pretrained('qwen_customer_support_lora')")

In [ ]:
# Opción 2: Guardar modelo completo mergeado (para deployment)
# Esto crea un modelo standalone sin necesidad de adaptadores

model.save_pretrained_merged(
    "qwen_customer_support_merged",
    tokenizer,
    save_method = "merged_16bit",  # O "merged_4bit" para menor tamaño
)

print("✅ Modelo completo mergeado guardado en: qwen_customer_support_merged/")
print("Este modelo puede ser cargado directamente con Transformers estándar")

In [ ]:
# Opción 3: Subir a Hugging Face Hub (opcional)
# Requiere: huggingface-cli login

# model.push_to_hub(
#     "tu-usuario/qwen-customer-support",
#     token = "tu_token_aqui"
# )
# tokenizer.push_to_hub(
#     "tu-usuario/qwen-customer-support",
#     token = "tu_token_aqui"
# )

print("ℹ️ Para subir a HF Hub, descomenta el código y agrega tu token")

## 10. Cargar Modelo Guardado

Ejemplo de cómo cargar el modelo que acabamos de guardar.

In [ ]:
# Cargar modelo con adaptadores LoRA
model_loaded, tokenizer_loaded = FastLanguageModel.from_pretrained(
    model_name = "qwen_customer_support_lora",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model_loaded)

print("✅ Modelo cargado exitosamente desde disco")

# Probar
test_prompt = alpaca_prompt.format(
    "Responde a la consulta del cliente de manera profesional y útil.",
    "¿Cómo cambio mi contraseña del WiFi?",
    ""
)
inputs = tokenizer_loaded([test_prompt], return_tensors="pt").to("cuda")
outputs = model_loaded.generate(**inputs, max_new_tokens=150)
print("\nRespuesta del modelo cargado:")
print(tokenizer_loaded.decode(outputs[0], skip_special_tokens=True).split("### Respuesta:")[1])

## 11. Ejercicios Prácticos

### Ejercicio 1: Mejorar el Dataset
Agrega 10 nuevos ejemplos de consultas reales que tus clientes harían. Entrena de nuevo y compara resultados.

### Ejercicio 2: Experimentar con Hiperparámetros
Prueba diferentes configuraciones:
- `r = 8` vs `r = 32` (rank de LoRA)
- `learning_rate = 1e-4` vs `5e-4`
- `num_train_epochs = 1` vs `5`

### Ejercicio 3: Evaluar Cuantitativamente
Crea un test set de 20 consultas con respuestas "gold standard" y calcula:
- Precisión (las respuestas son correctas?)
- BLEU score comparado con gold standard
- Tiempo de inferencia

### Ejercicio 4: Multi-turn Conversation
Modifica el código para manejar conversaciones de múltiples turnos (memoria de contexto).

### Ejercicio 5: Deployment
Crea una API REST con FastAPI que exponga tu modelo fine-tuned como servicio.

## Recursos Adicionales

- **Unsloth Docs**: https://github.com/unslothai/unsloth
- **LoRA Paper**: https://arxiv.org/abs/2106.09685
- **QLoRA Paper**: https://arxiv.org/abs/2305.14314
- **Qwen2.5 Models**: https://huggingface.co/Qwen
- **TRL Library**: https://github.com/huggingface/trl

## Conclusiones

En este taller has aprendido:
✅ Cargar y configurar Qwen2.5 con Unsloth  
✅ Aplicar LoRA para fine-tuning eficiente  
✅ Preparar datasets en formato Alpaca  
✅ Entrenar modelos en GPU consumidor (12-16GB)  
✅ Evaluar y comparar modelos base vs fine-tuned  
✅ Guardar y cargar modelos para producción  

**Próximo paso**: Lleva esto a producción con tu propio dataset y caso de uso! 🚀